# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`  
This notebook provides a step-by-step guide for loading and exploring the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{getattr(metadata, 'name', '<No name>')}: {getattr(metadata, 'description', '<No description>')}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. In Croissant, record sets organize data tables or main data structures; each contains fields referencing columns of the data.

Below, we will print out the available record sets, fields, and columns, using their `@id` to ensure precise references.

In [ ]:
# Discover all record sets
print("Available Record Sets:\n")
record_sets = list(dataset.record_sets)

for rs in record_sets:
    print(f"Record Set @id: {rs.id}")
    print(f"  Name: {getattr(rs, 'name', '<No name>')}")
    print(f"  Description: {getattr(rs, 'description', '<No description>')}")
    # List all fields (columns) in this record set
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    Field @id: {field.id} | Name: {getattr(field, 'name', '')} | DataType: {getattr(field, 'data_type', '')}")
    print('-'*60)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. 

We'll extract the main record set containing the participant survey data. Make sure to reference the record set and field by their exact `@id` discovered above. For demonstration, we use the first record set.

In [ ]:
# Use the primary record set, typically survey or tabular data
if len(record_sets) == 0:
    raise RuntimeError("No record sets found in the Croissant metadata.")

# Select the main data record set (the first one listed)
main_record_set_id = record_sets[0].id
print(f"Using record set: {main_record_set_id}\n")

# If more record sets are required, add their @ids here
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    # Load all records for this record set into a DataFrame
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for {rs_id}.")
        print(f"Columns: {dataframes[rs_id].columns.tolist()}")
        print(dataframes[rs_id].head())
    else:
        print(f"No records found for {rs_id}.")

# For EDA, use the main_record_set_id
df = dataframes.get(main_record_set_id)

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing: filtering, normalization, and grouping for further analysis. You can substitute the numeric and categorical field `@id`s with those seen in Section 2 or modify for your analysis scenario.

**Example analysis:**
- Select a numeric field such as GAD-7, PHQ-9, or PSQ total score.
- Filter for high scores, normalize, and group by an attribute such as gender or education level.

In [ ]:
# Edit these with the actual field @id as discovered in the Data Overview section.
if df is not None:
    # Let's try to find a numeric field likely to exist (search for GAD-7, PHQ-9, or 'score')
    numeric_field_candidates = [col for col in df.columns if any(t in col.lower() for t in ['gad7', 'phq9', 'psq', 'score', 'total'])]
    if numeric_field_candidates:
        numeric_field = numeric_field_candidates[0]
    else:
        # Fallback to the first float/int column
        numeric_types = df.select_dtypes(include=['number']).columns
        if len(numeric_types) > 0:
            numeric_field = numeric_types[0]
        else:
            raise RuntimeError('No numeric fields found for analysis.')
    print(f"Using numeric field: {numeric_field}")

    # Threshold for filtering (e.g., moderate symptoms)
    threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[numeric_field + '_normalized'] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

    # Attempt to group by a relevant field, e.g. gender, education, village, if available
    group_field_candidates = [col for col in df.columns if any(x in col.lower() for x in ['gender', 'sex', 'education', 'village', 'marital'])]
    group_field = group_field_candidates[0] if group_field_candidates else None
    if group_field and pd.api.types.is_string_dtype(df[group_field]):
        grouped_df = filtered_df.groupby(group_field)[numeric_field, numeric_field + '_normalized'].mean().reset_index()
        print(f"\nGrouped data by {group_field}:")
        print(grouped_df.head())
    else:
        print('No suitable group field found for grouping!')
else:
    print('No main dataframe loaded. Check earlier cells for errors.')

## 5. Visualization
Let's visualize the distribution of the selected numeric field and compare group means if grouping is available. Requires `matplotlib` and `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field in df.columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=15)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # Visualize group means if grouped_df exists
    if 'grouped_df' in locals() and group_field:
        plt.figure(figsize=(8, 4))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.show()
else:
    print('No data available for visualization.')

## 6. Conclusion
In this notebook, we've:
- Loaded the FAIR² Mental Health Survey Dataset via its Croissant schema using `mlcroissant`.
- Explored available record sets, fields, and columns by their `@id`.
- Extracted the main survey record set into a Pandas DataFrame and previewed the data.
- Applied basic exploratory data analysis: filtering, normalization, and grouping using demographic fields.
- Generated visualizations for the main numeric (e.g., symptom score) variables in the dataset.

This workflow can be further extended with advanced analysis, statistical modeling, and machine learning on AI-ready datasets described with Croissant schemas.

For more information, see the [Croissant specification](https://mlcommons.org/croissant/) and the [FAIR² Dataset description](https://sen.science/doi/10.71728/senscience.vcs2-05nj).